In [50]:
### import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from nltk.sentiment import SentimentIntensityAnalyzer

In [51]:
# read the dataset
df = pd.read_csv("data/test.csv")

In [52]:
df = df.drop_duplicates()
df = df.dropna()

In [53]:
df['body']

0       EnronOptions Announcement\n\n\nWe have updated...
1       Marc,\n\nUnfortunately, today is not going to ...
2       When: Wednesday, June 06, 2001 10:00 AM-11:00 ...
3       we were thinking papasitos (we can meet somewh...
4       Since you never gave me the $20 for the last t...
                              ...                        
2186    Thanks for the resume.  She has had some good ...
2187    Attached please find the following documents:\...
2188    Good to finally hear from.  Judging from your ...
2189    It looks like we have our 12 teams.  We will p...
2190    We will need this, so I am sending it to you a...
Name: body, Length: 2191, dtype: object

In [54]:
## delete '\n' string inside the message
df['body'] = df['body'].str.replace('\n', ' ')
df['body'] = df['body'].str.replace(r'\s+', ' ', regex=True).str.strip()
df['body'] = df['body'].str.replace('=20', ' ', regex=False)
df['body'] = df['body'].str.replace('=01', '', regex=False)

In [55]:
df['body']

0       EnronOptions Announcement We have updated the ...
1       Marc, Unfortunately, today is not going to wor...
2       When: Wednesday, June 06, 2001 10:00 AM-11:00 ...
3       we were thinking papasitos (we can meet somewh...
4       Since you never gave me the $20 for the last t...
                              ...                        
2186    Thanks for the resume. She has had some good e...
2187    Attached please find the following documents: ...
2188    Good to finally hear from. Judging from your e...
2189    It looks like we have our 12 teams. We will pr...
2190    We will need this, so I am sending it to you a...
Name: body, Length: 2191, dtype: object

### Task 1: Sentiment Labeling

In [56]:
analyzer = SentimentIntensityAnalyzer()

sentiment_label = []
for sentence in df['body']:
    vs = analyzer.polarity_scores(sentence)
    score = vs['compound']
    if score >= 0.05:
        sentiment_label.append('Positive')
    elif score <= -0.05:
        sentiment_label.append('Negative')
    else:
        sentiment_label.append('Neutral')

In [57]:
df['Sentiment'] = sentiment_label
df[['body','Sentiment']]

,body,Sentiment
0,EnronOptions Announcement We have updated the ...,Positive
1,"Marc, Unfortunately, today is not going to wor...",Positive
2,"When: Wednesday, June 06, 2001 10:00 AM-11:00 ...",Neutral
3,we were thinking papasitos (we can meet somewh...,Neutral
4,Since you never gave me the $20 for the last t...,Positive
...,...,...
2186,Thanks for the resume. She has had some good e...,Positive
2187,Attached please find the following documents: ...,Positive
2188,Good to finally hear from. Judging from your e...,Positive
2189,It looks like we have our 12 teams. We will pr...,Positive


In [58]:
print(f"The number of positive message(s): {(df['Sentiment'] == 'Positive').sum()}")
print(f"The number of neutral message(s): {(df['Sentiment'] == 'Neutral').sum()}")
print(f"The number of negative message(s): {(df['Sentiment'] == 'Negative').sum()}")

The number of positive message(s): 1528
The number of neutral message(s): 511
The number of negative message(s): 152


### Task 2 : EDA

In [59]:
print(f"There are {df.shape[0]} emails in this dataset.")

There are 2191 emails in this dataset.


In [60]:
# Create a variable to represent year only
df['Year'] = pd.to_datetime(df['date']).dt.year
df.head(5)

,Subject,body,date,from,Sentiment,Year
0,EnronOptions Update!,EnronOptions Announcement We have updated the ...,5/10/2010,sally.beck@enron.com,Positive,2010
1,(No Subject),"Marc, Unfortunately, today is not going to wor...",7/29/2010,eric.bass@enron.com,Positive,2010
2,Phone Screen Interview - Shannon L. Burnham,"When: Wednesday, June 06, 2001 10:00 AM-11:00 ...",7/25/2011,sally.beck@enron.com,Neutral,2011
3,RE: My new work email,we were thinking papasitos (we can meet somewh...,3/25/2010,johnny.palmer@enron.com,Neutral,2010
4,Bet,Since you never gave me the $20 for the last t...,5/21/2011,lydia.delgado@enron.com,Positive,2011


In [61]:
# Check the numbers of emails each year
df['Year'].value_counts()

Year
2011    1097
2010    1094
Name: count, dtype: int64

In [62]:
duplication = df.duplicated(subset=['body'])
print(f"There are {len(df[duplication])} duplicates email bodies.")

There are 682 duplicates email bodies.


In [63]:
df['from'].nunique()
## There are only 10 unique senders from 2010 to 2011

10

### Task 3: Employee Score Calculation

In [65]:
# Set score for each sentiment
score = {'Positive': 1, 'Neutral': 0, 'Negative': -1}

# Assign a score to each message
df['Score'] = df['Sentiment'].map(score)
df.head(10)

,Subject,body,date,from,Sentiment,Year,Score
0,EnronOptions Update!,EnronOptions Announcement We have updated the ...,5/10/2010,sally.beck@enron.com,Positive,2010,1
1,(No Subject),"Marc, Unfortunately, today is not going to wor...",7/29/2010,eric.bass@enron.com,Positive,2010,1
2,Phone Screen Interview - Shannon L. Burnham,"When: Wednesday, June 06, 2001 10:00 AM-11:00 ...",7/25/2011,sally.beck@enron.com,Neutral,2011,0
3,RE: My new work email,we were thinking papasitos (we can meet somewh...,3/25/2010,johnny.palmer@enron.com,Neutral,2010,0
4,Bet,Since you never gave me the $20 for the last t...,5/21/2011,lydia.delgado@enron.com,Positive,2011,1
5,RE: Favor,"sure, just call me the bank that delivers. we ...",10/23/2011,eric.bass@enron.com,Positive,2011,1
6,MG Inventory Summaries,Inventory summaries for both MGL and MGMCC as ...,4/5/2010,kayne.coulter@enron.com,Neutral,2010,0
7,Forgot the Attachment,Please print attachment and make sure that e:m...,4/21/2010,patti.thompson@enron.com,Positive,2010,1
8,Garvin Brown - AXIA Sr. Power Scheduler,Please advise me of your interest in Garvin's ...,2/7/2010,sally.beck@enron.com,Positive,2010,1
9,More Dallas ASE Information,The start time for Tuesday morning has been ch...,2/6/2010,kayne.coulter@enron.com,Negative,2010,-1


In [78]:
df['date'] = pd.to_datetime(df['date'])
df['Month'] = df['date'].dt.to_period('M')
df['Month']

0       2010-05
1       2010-07
2       2011-07
3       2010-03
4       2011-05
         ...   
2186    2011-06
2187    2011-01
2188    2011-01
2189    2011-03
2190    2010-10
Name: Month, Length: 2191, dtype: period[M]

In [83]:
monthly_scores = df.groupby(['from', 'Month'])['Score'].sum()
monthly_scores = monthly_scores.reset_index()
monthly_scores = monthly_scores.rename(columns={'from': 'Employee', 'Score': 'Monthly Sentiment Score'})
monthly_scores.head(20)

,Employee,Month,Monthly Sentiment Score
0,bobette.riner@ipgdirect.com,2010-01,1
1,bobette.riner@ipgdirect.com,2010-02,7
2,bobette.riner@ipgdirect.com,2010-03,6
3,bobette.riner@ipgdirect.com,2010-04,3
4,bobette.riner@ipgdirect.com,2010-05,2
5,bobette.riner@ipgdirect.com,2010-06,2
6,bobette.riner@ipgdirect.com,2010-07,8
7,bobette.riner@ipgdirect.com,2010-08,4
8,bobette.riner@ipgdirect.com,2010-09,2
9,bobette.riner@ipgdirect.com,2010-10,6


In [89]:
for employee in monthly_scores['Employee'].unique():
    print(f"{employee}")
    employee_name = monthly_scores[monthly_scores['Employee'] == employee]
    print(employee_name[['Month', 'Monthly Sentiment Score']].head(5))

bobette.riner@ipgdirect.com
     Month  Monthly Sentiment Score
0  2010-01                        1
1  2010-02                        7
2  2010-03                        6
3  2010-04                        3
4  2010-05                        2
don.baughman@enron.com
      Month  Monthly Sentiment Score
24  2010-01                        5
25  2010-02                        6
26  2010-03                        2
27  2010-04                        9
28  2010-05                       16
eric.bass@enron.com
      Month  Monthly Sentiment Score
48  2010-01                        9
49  2010-02                        2
50  2010-03                        4
51  2010-04                        2
52  2010-05                        6
john.arnold@enron.com
      Month  Monthly Sentiment Score
72  2010-01                        5
73  2010-02                       11
74  2010-03                        7
75  2010-04                        8
76  2010-05                        3
johnny.palmer@enron.com
 

### Task 4: Employee Ranking